In [1]:
import tiktoken

In [2]:
tokenizer = tiktoken.get_encoding("gpt2")

with open("../tokenizer/the-verdict.txt", "r", encoding="utf-8") as f:
    text = f.read()

enc_text = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

In [3]:
print(len(enc_text))
enc_sample = enc_text[50:]

5145


In [4]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
print(f"x: {x}")
print(f"y: {y}")

x: [290, 4920, 2241, 287]
y: [4920, 2241, 287, 257]


In [5]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "----->", desired)

[290] -----> 4920
[290, 4920] -----> 2241
[290, 4920, 2241] -----> 287
[290, 4920, 2241, 287] -----> 257


In [6]:
import torch
from torch.utils.data import DataLoader, Dataset

In [7]:
class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text)

        for i in range(0, len(token_ids)-max_length, stride):
            input_chunks = token_ids[i:i+max_length]
            target_chunks = token_ids[i + 1 : i + 1 + max_length]
            self.input_ids.append(torch.tensor(input_chunks))
            self.target_ids.append(torch.tensor(target_chunks))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

In [8]:
def create_dataloader_v1(text, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(text, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset, 
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader


In [9]:
with open("../tokenizer/the-verdict.txt", "r", encoding="utf-8") as f:
    text = f.read()

dataloader = create_dataloader_v1(text, batch_size=8, max_length=4, stride=4, shuffle=False)


In [10]:
data = iter(dataloader)

In [11]:
first_batch = next(data)
print(first_batch)

[tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]]), tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])]


In [12]:
second_batch = next(data)
second_batch

[tensor([[  287,   262,  6001,   286],
         [  465, 13476,    11,   339],
         [  550,  5710,   465, 12036],
         [   11,  6405,   257,  5527],
         [27075,    11,   290,  4920],
         [ 2241,   287,   257,  4489],
         [   64,   319,   262, 34686],
         [41976,    13,   357, 10915]]),
 tensor([[  262,  6001,   286,   465],
         [13476,    11,   339,   550],
         [ 5710,   465, 12036,    11],
         [ 6405,   257,  5527, 27075],
         [   11,   290,  4920,  2241],
         [  287,   257,  4489,    64],
         [  319,   262, 34686, 41976],
         [   13,   357, 10915,   314]])]

## creating token embeddings

In [13]:
input_ids = torch.tensor([1,2,3,4])

vocab_size = 6
output_dim = 3


In [14]:
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [15]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [16]:
input_dim = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(input_dim, output_dim)

In [17]:
max_length = 4
dataloader = create_dataloader_v1(text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter = iter(dataloader)

In [18]:
first_input, first_target = next(data_iter)
print(first_input)
print(first_target)

tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [19]:
token_embedding = token_embedding_layer(first_input)
print(token_embedding.shape)

torch.Size([8, 4, 256])


In [20]:
## position embedding
context_lentgh = max_length
pos_embedding_layer = torch.nn.Embedding(context_lentgh, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_lentgh))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [21]:
input_embeddings = token_embedding + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
